# CAPFG 14 — Reviewer-minimal retrospective analysis (NO retraining)

**Goal.** Reuse the original **CAPFG10 DINOv2** and **CAPFG11 ResNet-50** confirmatory outputs to address IJITCS review comments with **zero backbone extraction and zero classifier fitting**.

This notebook computes:
1. Data completeness / task audit, actual support-fitted PCA dimensions.
2. Descriptive results by **dataset × shot × backbone × method** and paired subgroup effects.
3. Worst-task and negative-effect analysis, plus available DINOv2 external lightweight baselines.
4. Frequencies of selected `lambda_syn` and `C`, selection-degeneracy rate, stage sizes.
5. **Retrospective validation-only** tolerance sensitivity based on original candidate validation scores.

**Important:** Alternative tolerance test metrics **cannot** be inferred from candidate validation scores. This analysis never claims a newly selected configuration was tested. This notebook does not prove superiority over original Distribution Calibration [16] or Learnable Distribution Calibration [17].

The original settings remain frozen. Post-hoc sensitivity is labeled as reviewer-requested analysis, NOT a new confirmatory protocol.

## 1. Attach existing outputs (Kaggle Input)

Attach original files from **CAPFG10** and **CAPFG11** as Kaggle datasets, preferably these small CSVs:

- `confirmatory_task_results.csv`, `confirmatory_candidate_results.csv`, `confirmatory_protocol_audit.csv`.
- `robustness_task_results.csv`, `robustness_candidate_results.csv`, `robustness_protocol_audit.csv`.

Alternatively, attach the original `CAPFG10_confirmatory_package.zip` and `CAPFG11_backbone_robustness_summary_package.zip` (the CAPFG10 ZIP may be large because it also contains per-image predictions). This notebook reads only the six small CSVs, directly from ZIP when possible.

**Do NOT rerun Notebook 10/11 merely to generate this analysis** if these outputs are already available from prior Kaggle runs. If one file is missing, the notebook raises a clear error; do not silently replace old results with fabricated or recreated values.

In [ ]:
from pathlib import Path
import io, json, zipfile, sys, os, warnings
import numpy as np
import pandas as pd
from collections import defaultdict

SEARCH_ROOTS = [Path('/kaggle/input'), Path('/kaggle/working'), Path('/mnt/data'), Path.cwd()]
OUT = Path('/kaggle/working/CAPFG14_revision_minimal') if Path('/kaggle/working').exists() else Path.cwd()/'CAPFG14_revision_minimal'
OUT.mkdir(parents=True, exist_ok=True)
# Optionally override any filename with an exact path here:
MANUAL_FILES = {
    'confirmatory_task_results.csv': None,
    'confirmatory_candidate_results.csv': None,
    'confirmatory_protocol_audit.csv': None,
    'robustness_task_results.csv': None,
    'robustness_candidate_results.csv': None,
    'robustness_protocol_audit.csv': None,
}
print('Python:', sys.version.split()[0], 'Pandas:', pd.__version__)
print('Output directory:', OUT)

In [ ]:
# Build a small index of CSVs and ZIPs; avoids scanning any image bytes.
paths = []
for root in SEARCH_ROOTS:
    if root.exists():
        for suffix in ('*.csv', '*.zip'):
            try:
                paths.extend(root.rglob(suffix))
            except (OSError, PermissionError):
                pass
paths = sorted(set(p.resolve() for p in paths if p.is_file()))
zip_index = {}
for p in paths:
    if p.suffix.lower() != '.zip':
        continue
    try:
        with zipfile.ZipFile(p) as z:
            zip_index[p] = [x for x in z.namelist() if x.lower().endswith('.csv')]
    except (zipfile.BadZipFile,OSError,PermissionError):
        continue

def find_csv(name, required=True):
    exact=MANUAL_FILES.get(name)
    if exact:
        p=Path(exact)
        if not p.is_file(): raise FileNotFoundError(str(p))
        print(name,'<- direct explicit',p)
        return pd.read_csv(p)
    # Prefer an uncompressed original file for faster access.
    candidates = [p for p in paths if p.suffix.lower()=='.csv' and p.name==name]
    if candidates:
        p=candidates[0]
        print(name,'<- direct',p)
        return pd.read_csv(p)
    for z,items in zip_index.items():
        matches=[item for item in items if Path(item).name==name]
        if matches:
            with zipfile.ZipFile(z) as archive:
                with archive.open(matches[0]) as f:
                    data=pd.read_csv(f)
            print(name,'<- zip',z,'member',matches[0]);return data
    if required: raise FileNotFoundError(f'Missing {name}. Attach original CSV or output ZIP from CAPFG10/CAPFG11.')
    print('[OPTIONAL MISSING]', name)
    return pd.DataFrame()

raw = {}
for tag, prefix in [('DINOv2-Small','confirmatory'),('ResNet-50','robustness')]:
    raw[tag]={key:find_csv(f'{prefix}_{key}.csv',required=(key!='protocol_audit'))
             for key in ['task_results','candidate_results','protocol_audit']}
print({k:{x:df.shape for x,df in value.items()} for k,value in raw.items()})

## 2. Harmonize schemas and audit original experiment coverage

In [ ]:
EXPECTED_TASKS=60
MAIN=['LR-Real','PFG-UnweightedLR','PFG-NW-F1Only','CA-PFG-MO']
SEEDS=[137,139,149,151,157,163,167,173,179,181]

def harmonize(df,backbone):
    out=df.copy(); out['backbone']=backbone
    if 'task_key' not in out: raise ValueError(f'{backbone}: missing task_key')
    out['task_key']=out['task_key'].astype(str)
    if 'dataset' not in out:
        out['dataset']=out['task_key'].str.extract(r'^(.+?)__k\d+__seed\d+')[0]
    if 'K' not in out:
        out['K']=pd.to_numeric(out['task_key'].str.extract(r'__k(\d+)__')[0],errors='coerce')
    if 'seed' not in out:
        out['seed']=pd.to_numeric(out['task_key'].str.extract(r'__seed(\d+)')[0],errors='coerce')
    out['K']=pd.to_numeric(out['K'],errors='coerce')
    out['seed']=pd.to_numeric(out['seed'],errors='coerce')
    return out
T=pd.concat([harmonize(raw[x]['task_results'],x) for x in raw],ignore_index=True)
Q=pd.concat([harmonize(raw[x]['candidate_results'],x) for x in raw],ignore_index=True)
if 'method_pool' not in Q and 'method' not in Q: raise ValueError('Candidate method identifier absent')
Q['pool'] = Q['method_pool'] if 'method_pool' in Q else Q['method']
if 'method_pool' in Q and 'method' in Q: Q['pool']=Q['pool'].fillna(Q['method'])
Q['pool']=Q['pool'].astype(str)
for c in ['C','lambda_syn','validation_macro_f1','validation_nll','validation_ece']:
    if c not in Q: raise ValueError('Missing candidate field: '+c)
    Q[c]=pd.to_numeric(Q[c],errors='coerce')
for c in ['test_macro_f1','test_nll','test_ece','test_brier']:
    if c not in T: raise ValueError('Missing test metric '+c)
    T[c]=pd.to_numeric(T[c],errors='coerce')

checks=[]
for b in raw:
    t=T[(T.backbone==b)&(T.method.isin(MAIN))]
    q=Q[(Q.backbone==b)&(Q.pool=='PFG-NW')]
    count=t.groupby('task_key')['method'].nunique()
    n_task=t.task_key.nunique()
    cq=q.groupby('task_key')[['C','lambda_syn']].size()
    g=t.groupby('task_key')['K'].first()
    audit=raw[b]['protocol_audit']
    checks.append(dict(backbone=b,main_task_count=n_task,expected_tasks=EXPECTED_TASKS,
        main_rows=len(t),expected_main_rows=4*EXPECTED_TASKS,
        tasks_all_four_methods=int((count==4).sum()),
        pfg_nw_candidate_rows=len(q),expected_candidate_rows=12*EXPECTED_TASKS,
        tasks_with_12_pfg_candidates=int((cq==12).sum()),
        task_keys_5shot=int((g==5).sum()),
        candidate_validation_finite=bool(np.isfinite(q[['validation_macro_f1','validation_nll','validation_ece']].to_numpy()).all()),
        protocol_audit_rows=len(audit),
        pca_components_unique=';'.join(map(str,sorted(t.pca_components.dropna().unique()))) if 'pca_components' in t else 'unavailable'))
audit_summary=pd.DataFrame(checks)
audit_summary.to_csv(OUT/'01_protocol_audit_summary.csv',index=False)
print(audit_summary.to_string(index=False))
assert all(audit_summary.main_task_count==EXPECTED_TASKS),'TASK COUNT mismatch: inspect source data'
assert all(audit_summary.tasks_all_four_methods==EXPECTED_TASKS),'Missing main methods'
assert all(audit_summary.tasks_with_12_pfg_candidates==EXPECTED_TASKS),'Missing PFG-NW candidate grid'

pca=T.groupby(['backbone','dataset','K']).agg(n_tasks=('task_key','nunique'),
        min_pca=('pca_components','min'),max_pca=('pca_components','max'),
        unique_pca=('pca_components',lambda x: ';'.join(map(str, sorted(set(x.dropna().astype(int))))))).reset_index()
pca.to_csv(OUT/'02_actual_pca_dimensions.csv',index=False)
print('\nActual PCA components (from original recorded results):')
print(pca.to_string(index=False))

## 3. Existing metrics: by dataset, K and backbone (descriptive; no new model fitting)

In [ ]:
metrics=['macro_f1','nll','ece','brier']
main=T[T.method.isin(MAIN)].copy()
group_cols=['backbone','dataset','K','method']
agg_dict={'task_key':'nunique'}
agg_dict.update({f'test_{m}':['mean','std'] for m in metrics})
summ=main.groupby(group_cols).agg(agg_dict)
summ.columns=['n_tasks' if k=='task_key' else k.replace('test_','')+'_'+v for k,v in summ.columns]
summ=summ.reset_index()
summ.to_csv(OUT/'03_results_by_dataset_shot_backbone.csv',index=False)
print(summ.head(25).to_string(index=False))

# Signed effects: Macro-F1 challenger minus reference; NLL/ECE/Brier reference minus challenger.
comparisons=[('unweighted_vs_real','PFG-UnweightedLR','LR-Real'),
 ('normalized_vs_unweighted','PFG-NW-F1Only','PFG-UnweightedLR'),
 ('calibration_aware_vs_f1_only','CA-PFG-MO','PFG-NW-F1Only')]
fx=[]; per_task=[]
for label,challenger,reference in comparisons:
    c=main[main.method==challenger].set_index(['backbone','task_key'])
    r=main[main.method==reference].set_index(['backbone','task_key'])
    ix=c.index.intersection(r.index)
    assert len(ix)==2*EXPECTED_TASKS,f'Unmatched {label}: {len(ix)}'
    for (b,k) in ix:
        z={'contrast':label,'backbone':b,'task_key':k,'dataset':c.loc[(b,k),'dataset'],'K':int(c.loc[(b,k),'K'])}
        for m in metrics:
            a=float(c.loc[(b,k),'test_'+m]);v=float(r.loc[(b,k),'test_'+m]);
            z['favorable_'+m]=(a-v) if m=='macro_f1' else (v-a)
        per_task.append(z)
per_task=pd.DataFrame(per_task)
per_task.to_csv(OUT/'04_paired_task_effects.csv',index=False)
for keys,grp in per_task.groupby(['contrast','backbone','dataset','K']):
    for m in metrics:
        d=grp['favorable_'+m].values
        fx.append(dict(zip(['contrast','backbone','dataset','K'],keys),metric=m,n=len(d),
         paired_effect_mean=float(np.mean(d)),positive=int(sum(d>1e-12)),
         negative=int(sum(d< -1e-12)),ties=int(sum(abs(d)<=1e-12))))
fx=pd.DataFrame(fx);fx.to_csv(OUT/'05_paired_effects_by_dataset_shot.csv',index=False)
print('\nPaired subgroup mean effects (positive = favorable):')
print(fx[fx.metric.isin(['macro_f1','nll','ece'])].head(35).to_string(index=False))

## 4. Failure tasks and prior baseline coverage

In [ ]:
worst=(main[main.method=='CA-PFG-MO']
    .sort_values(['backbone','dataset','test_macro_f1','K','seed'])
    .groupby(['backbone','dataset'],group_keys=False).head(10))
worst.to_csv(OUT/'06_worst_ca_pfg_tasks.csv',index=False)
print('Worst 10 CA-PFG-MO tasks per dataset/backbone:')
print(worst[['backbone','dataset','K','seed','task_key','test_macro_f1','test_nll','test_ece']].to_string(index=False))

negative=per_task[(per_task.contrast.isin(['normalized_vs_unweighted','calibration_aware_vs_f1_only']))&
                  (per_task.favorable_macro_f1< -1e-12)].copy()
negative.to_csv(OUT/'07_negative_effect_tasks.csv',index=False)
print('\nNumber of negative Macro-F1 effects:')
print(negative.groupby(['contrast','backbone','dataset','K']).size().to_string())

available=sorted(T.method.dropna().unique())
print('\nAll method labels found in original results: ',available)
extra=T[~T.method.isin(MAIN)]
if len(extra):
    extra_group=extra.groupby(['backbone','method']).agg(n_tasks=('task_key','nunique'),
       macro_f1_mean=('test_macro_f1','mean'),nll_mean=('test_nll','mean'),ece_mean=('test_ece','mean')).reset_index()
    extra_group.to_csv(OUT/'08_additional_existing_baselines_and_ablations.csv',index=False)
    print(extra_group.to_string(index=False))
else:
    print('No additional baseline or ablation methods found in the original package')

## 5. Selection frequencies, matching and post-hoc validation-only tolerance sensitivity

This section **does not retrain** any model, nor does it calculate unobserved test outcomes for alternative selected candidates. The published selector uses $(\delta_F,\delta_N,\delta_E)=(0.002,0.01,0.01)$ and tie-breaks on lower `lambda_syn`, then lower `C`.

The reviewer-requested sensitivity grid changes **one tolerance at a time**, leaving the two others at their original values. All metrics reported here are **validation-only**.

In [ ]:
ORIGINAL=(0.002,0.01,0.01)
SCENARIOS=[('original',*ORIGINAL),
 ('F_0',0.0,0.01,0.01),('F_0.001',0.001,0.01,0.01),('F_0.005',0.005,0.01,0.01),
 ('N_0',0.002,0.0,0.01),('N_0.005',0.002,0.005,0.01),('N_0.020',0.002,0.020,0.01),
 ('E_0',0.002,0.01,0.0),('E_0.005',0.002,0.01,0.005),('E_0.020',0.002,0.01,0.020)]

def choose(q,df,dn,de):
    # Reproduce code in CAPFG10/11: max F1 -> tolerance -> min NLL -> tolerance -> min ECE -> tolerance.
    fmax=q.validation_macro_f1.max()
    hf=q[q.validation_macro_f1>=fmax-df-1e-14]
    nmin=hf.validation_nll.min()
    hn=hf[hf.validation_nll<=nmin+dn+1e-14]
    emin=hn.validation_ece.min()
    he=hn[hn.validation_ece<=emin+de+1e-14]
    selected=he.sort_values(['lambda_syn','C'],kind='mergesort').iloc[0]
    return selected, (len(q),len(hf),len(hn),len(he))

def choose_f1(q):
    best=q.validation_macro_f1.max()
    return q[q.validation_macro_f1 >= best-1e-12].sort_values(['lambda_syn','C'],kind='mergesort').iloc[0]

def same(a,b): return abs(float(a.lambda_syn)-float(b.lambda_syn))<1e-12 and abs(float(a.C)-float(b.C))<1e-12

records=[]
for (b,key),q in Q[Q.pool=='PFG-NW'].groupby(['backbone','task_key']):
    if len(q)!=12: raise ValueError(f'{b}/{key}: expected 12 normalized candidates, got {len(q)}')
    baseline,_=choose(q,*ORIGINAL)
    f1=choose_f1(q)
    meta=q.iloc[0]
    for scenario,df,dn,de in SCENARIOS:
        chosen,sizes=choose(q,df,dn,de)
        records.append(dict(backbone=b,task_key=key,dataset=meta.dataset,K=int(meta.K),seed=int(meta.seed),
           scenario=scenario,delta_F=df,delta_N=dn,delta_E=de,
           selected_lambda_syn=float(chosen.lambda_syn),selected_C=float(chosen.C),
           selected_validation_f1=float(chosen.validation_macro_f1),
           selected_validation_nll=float(chosen.validation_nll),
           selected_validation_ece=float(chosen.validation_ece),
           f1_only_validation_f1=float(f1.validation_macro_f1),
           f1_only_validation_nll=float(f1.validation_nll),
           f1_only_validation_ece=float(f1.validation_ece),
           same_as_original=int(same(chosen,baseline)),same_as_f1_only=int(same(chosen,f1)),
           n_all=sizes[0],n_after_f1=sizes[1],n_after_nll=sizes[2],n_after_ece=sizes[3]))
sel=pd.DataFrame(records)
sel.to_csv(OUT/'09_selection_sensitivity_task_level_VALIDATION_ONLY.csv',index=False)
sel['validation_f1_cost_vs_f1_only']=sel.f1_only_validation_f1-sel.selected_validation_f1
sel['validation_nll_improvement_vs_f1_only']=sel.f1_only_validation_nll-sel.selected_validation_nll
sel['validation_ece_improvement_vs_f1_only']=sel.f1_only_validation_ece-sel.selected_validation_ece
sensitivity=sel.groupby(['backbone','scenario','delta_F','delta_N','delta_E']).agg(
    n_tasks=('task_key','nunique'),pct_same_as_original=('same_as_original','mean'),
    pct_same_as_f1_only=('same_as_f1_only','mean'),
    avg_after_f1=('n_after_f1','mean'),avg_after_nll=('n_after_nll','mean'),avg_after_ece=('n_after_ece','mean'),
    validation_f1_cost_mean=('validation_f1_cost_vs_f1_only','mean'),
    validation_nll_benefit_mean=('validation_nll_improvement_vs_f1_only','mean'),
    validation_ece_benefit_mean=('validation_ece_improvement_vs_f1_only','mean')).reset_index()
sensitivity.to_csv(OUT/'10_sensitivity_summary_VALIDATION_ONLY.csv',index=False)
original=sel[sel.scenario=='original'].copy()
frequencies=original.groupby(['backbone','selected_lambda_syn','selected_C']).size().rename('n_tasks').reset_index()
frequencies['pct_tasks']=frequencies.n_tasks/frequencies.groupby('backbone').n_tasks.transform('sum')
frequencies.to_csv(OUT/'11_selected_config_frequencies.csv',index=False)

# Check reconstructed selector against original recorded selections, without looking at test scores.
selected_in_T=T[T.method=='CA-PFG-MO'][['backbone','task_key','selected_lambda_syn','selected_C']].copy()
validation=original.merge(selected_in_T,on=['backbone','task_key'],how='left',suffixes=('_reconstructed','_original'))
validation['match']=(np.isclose(validation.selected_lambda_syn_reconstructed,validation.selected_lambda_syn_original,atol=1e-10)&
   np.isclose(validation.selected_C_reconstructed,validation.selected_C_original,atol=1e-10))
validation.to_csv(OUT/'12_selection_reconstruction_audit.csv',index=False)
print('Selection reconstruction match:',validation['match'].sum(),'/',len(validation))
assert validation['match'].all(), 'Selection mismatch: inspect candidate metrics/ties, do not report sensitivity before reconciling.'
print('\nOriginal config frequencies:')
print(frequencies.to_string(index=False))
print('\nSensitivity summaries (VALIDATION only):')
print(sensitivity[['backbone','scenario','n_tasks','pct_same_as_original','pct_same_as_f1_only','avg_after_f1','validation_f1_cost_mean','validation_nll_benefit_mean','validation_ece_benefit_mean']].to_string(index=False))

## 6. Export compact upload package

The ZIP includes only small derived CSV tables, a manifest and a readable memo. Send **one output ZIP** to ChatGPT for review; keep the original Notebook 10/11 outputs unchanged. If prediction-level files are later needed for per-class analysis, send selected task `.parquet` separately.

In [ ]:
import zipfile
manifest={'source':'original CAPFG10 and CAPFG11 outputs; no model retraining',
          'original_tolerances':{'delta_F':0.002,'delta_N':0.01,'delta_E':0.01},
          'original_candidate_grid':{'lambda_syn':[0.125,0.25,0.5,1.0], 'C':[10.0,30.0,100.0]},
          'important_restriction':'Alternative tolerance selections are analyzed on validation data only; no alternative test predictions are available.',
          'n_original_tasks_per_backbone':EXPECTED_TASKS,
          'files':[p.name for p in sorted(OUT.glob('*.csv'))]}
(OUT/'README_results.json').write_text(json.dumps(manifest,indent=2),encoding='utf-8')
summary=['# CAPFG14: reviewer retrospective analyses (NO retraining)',
         'Original data source: CAPFG10/CAPFG11 task- and candidate-level outputs.',
         'New tolerance analyses are post-hoc validation-only; no novel test results are claimed.',
         '\n## Core coverage',audit_summary.to_string(index=False),
         '\n## PCA dimensions',pca.to_string(index=False),
         '\n## Selected configuration frequencies',frequencies.to_string(index=False),
         '\n## Validation tolerance sensitivity',sensitivity.to_string(index=False)]
(OUT/'REVIEWER_SUMMARY.md').write_text('\n\n'.join(summary),encoding='utf-8')
zip_path=OUT.parent/'CAPFG14_Reviewer_Minimal_Analysis_Results.zip'
with zipfile.ZipFile(zip_path,'w',compression=zipfile.ZIP_DEFLATED) as z:
    for p in sorted(OUT.iterdir()):
        if p.is_file(): z.write(p,arcname=p.name)
print('\nSAVED:',zip_path)
print('FILES:', sorted(p.name for p in OUT.iterdir()))
print('Please upload the ZIP to ChatGPT, plus original NB10/NB11 result ZIPs if available.')

In [ ]:
# Final cell — ZIP all reviewer-analysis outputs for download
from pathlib import Path
from zipfile import ZipFile, ZIP_DEFLATED

result_dir = Path(OUT)  # Defined in Cell 2
if not result_dir.exists():
    raise FileNotFoundError(f"Result folder not found: {result_dir}. Run previous cells first.")

result_files = sorted(p for p in result_dir.rglob("*") if p.is_file())
if not result_files:
    raise RuntimeError("No result files to compress. Run the analysis cells first.")

zip_file = Path('/kaggle/working/CAPFG14_Reviewer_Minimal_Analysis_Results.zip') if Path('/kaggle/working').exists() else result_dir.parent / 'CAPFG14_Reviewer_Minimal_Analysis_Results.zip'
with ZipFile(zip_file, 'w', compression=ZIP_DEFLATED) as archive:
    for file in result_files:
        archive.write(file, arcname=file.relative_to(result_dir))

print(f"ZIP created: {zip_file}")
print(f"Files included: {len(result_files)}")
print(f"Size: {zip_file.stat().st_size / 1024:.1f} KB")
print("Download the ZIP from Kaggle's Output /kaggle/working and upload it to ChatGPT.")
